In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

catalog_name= 'ecommerce'

### BRANDS TABLE PROCESSING

In [0]:
df_bronze= spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

+----------+-----------+-------------+--------------------+--------------------+
|brand_code| brand_name|category_code|        _source_file|         ingested_at|
+----------+-----------+-------------+--------------------+--------------------+
|      ACME|   AcmeTech|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      NOVW|  NovaWave |           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ZNTH|     Zenith|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      BYTM|    ByteMax|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ECOT|    EcoTone|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      SKYL|    SkyLink|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|     VOLT@|   VoltEdge|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      PHTX|   Photonix|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      URTL| UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      COTC| CottonClub|    

In [0]:
df_silver= df_bronze.withColumn("brand_name", F.trim(F.col("brand_name")))
df_silver.show(5)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
+----------+----------+-------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver=df_silver.withColumn("brand_code" , F.regexp_replace(F.col("brand_code"), r'[^A-Za-z0-9]' , ''))
df_silver.show(10)

+----------+----------+-------------+--------------------+--------------------+
|brand_code|brand_name|category_code|        _source_file|         ingested_at|
+----------+----------+-------------+--------------------+--------------------+
|      ACME|  AcmeTech|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      NOVW|  NovaWave|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ZNTH|    Zenith|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      BYTM|   ByteMax|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      ECOT|   EcoTone|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      SKYL|   SkyLink|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      VOLT|  VoltEdge|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      PHTX|  Photonix|           CE|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      URTL|UrbanTrail|          APP|dbfs:/Volumes/eco...|2026-02-08 11:00:...|
|      COTC|CottonClub|          APP|dbf

In [0]:
df_silver.select("category_code").distinct().show()

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|        BOOKS|
|          BKS|
|      GROCERY|
|         GRCY|
|          TOY|
|         TOYS|
|          SPT|
+-------------+



In [0]:
df_silver =df_silver.withColumn("category_code" , F.when(F.col("category_code")=="BOOKS" , "BKS")\
                                                 .when(F.col("category_code")=="GROCERY", "GRCY")\
                                                 .when(F.col("category_code")=="TOYS","TOY")
                                                 .otherwise(F.col("category_code")) 
                                                )

df_silver.select(F.col("category_code")).distinct().show()                                               

+-------------+
|category_code|
+-------------+
|           CE|
|          APP|
|          HNK|
|          BPC|
|          BKS|
|         GRCY|
|          TOY|
|          SPT|
+-------------+



In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

### CATEGORY TABLE PROCESSING

In [0]:
df_bronze= spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show()

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_path|         ingested_at|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          hnk|      Home & Kitchen|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          bpc|Beauty & Personal...|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          bks|               Books|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|         grcy|             Grocery|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          toy|        Toys & Games|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          spt|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|         grcy|             Grocery|dbfs:/Volumes/ec

In [0]:
df_silver =df_bronze.dropDuplicates(['category_code'])
df_silver.show()


+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_path|         ingested_at|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          app|             Apparel|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          hnk|      Home & Kitchen|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          bpc|Beauty & Personal...|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          bks|               Books|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|         grcy|             Grocery|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          toy|        Toys & Games|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          spt|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
+-------------+--------------------+--------------------+--------------------+



In [0]:
df_silver=df_silver.withColumn("category_code" , F.upper(F.col("category_code")))
df_silver.show()

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _source_path|         ingested_at|
+-------------+--------------------+--------------------+--------------------+
|           CE|         Electronics|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          APP|             Apparel|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          HNK|      Home & Kitchen|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          BPC|Beauty & Personal...|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          BKS|               Books|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|         GRCY|             Grocery|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          TOY|        Toys & Games|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
|          SPT|   Sports & Outdoors|dbfs:/Volumes/eco...|2026-02-08 11:17:...|
+-------------+--------------------+--------------------+--------------------+



In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_category")    

### PRODUCTS TABLE PROCESSING

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-02-08T11:20:20.488Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-02-08T11:20:20.488Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-02-08T11:20:20.488Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-02-08T11:20:20.488Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/source_data/raw/products/products.csv,2026-02-08T11:20:20.488Z


In [0]:
df_silver=df_bronze.withColumn("weight_grams", F.regexp_replace(F.col("weight_grams"),"g","").cast(IntegerType()))
df_silver=df_silver.withColumn("length_cm" , F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType()))
df_silver=df_silver.withColumn("category_code" , F.upper(F.col("category_code"))).withColumn("brand_code",F.upper(F.col("brand_code")))


df_silver.select("weight_grams","length_cm","category_code","brand_code").show(5)

+------------+---------+-------------+----------+
|weight_grams|length_cm|category_code|brand_code|
+------------+---------+-------------+----------+
|         305|     22.2|          HNK|      STCR|
|         682|     18.2|          HNK|      HMNS|
|         243|     18.2|           CE|      NOVW|
|         225|     17.6|          APP|      URTL|
|         455|     27.2|         GRCY|      GGRN|
+------------+---------+-------------+----------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df_silver.select("material").distinct().show()    

+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  
)


df_silver.select("weight_grams","length_cm","category_code","brand_code","rating_count").show(5)

+------------+---------+-------------+----------+------------+
|weight_grams|length_cm|category_code|brand_code|rating_count|
+------------+---------+-------------+----------+------------+
|         305|     22.2|          HNK|      STCR|           0|
|         682|     18.2|          HNK|      HMNS|           1|
|         243|     18.2|           CE|      NOVW|           0|
|         225|     17.6|          APP|      URTL|          50|
|         455|     27.2|         GRCY|      GGRN|           4|
+------------+---------+-------------+----------+------------+
only showing top 5 rows


In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

### CUSTOMERS TABLE PROCESSING

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")
df_bronze.show(5)

total= df_bronze.count()
print(total)

+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|  country|state|           file_name|    ingest_timestamp|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|    India|   MH|dbfs:/Volumes/eco...|2026-02-08 11:22:...|
|CUST000000000002|619489725433.0|          AU|Australia|  VIC|dbfs:/Volumes/eco...|2026-02-08 11:22:...|
|CUST000000000003|919390066524.0|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-02-08 11:22:...|
|CUST000000000004|917073741793.0|          IN|    India|   TN|dbfs:/Volumes/eco...|2026-02-08 11:22:...|
|CUST000000000005|618478772532.0|          AU|Australia|   WA|dbfs:/Volumes/eco...|2026-02-08 11:22:...|
+----------------+--------------+------------+---------+-----+--------------------+--------------------+
only showing top 5 rows
300000


In [0]:
print(df_bronze.filter(F.col("customer_id").isNull()).count())

300


In [0]:
df_silver=df_bronze.dropna(subset=["customer_id"])
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")


df_silver=df_silver.fillna("Not Available",subset=["phone"])
df_silver.select("customer_id","phone").show(10)

Row count after droping null values: 299700
+----------------+--------------+
|     customer_id|         phone|
+----------------+--------------+
|CUST000000000001|917280033536.0|
|CUST000000000002|619489725433.0|
|CUST000000000003|919390066524.0|
|CUST000000000004|917073741793.0|
|CUST000000000005|618478772532.0|
|CUST000000000006|916441718520.0|
|CUST000000000007| Not Available|
|CUST000000000008|446806361276.0|
|CUST000000000009| 18191801729.0|
|CUST000000000010| Not Available|
+----------------+--------------+
only showing top 10 rows


In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

### DATE TABLE PROCESSING

In [0]:
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")
df_bronze.show(5)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|  friday|      3|         -31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|02-08-2025|2025|SATURDAY|      3|         -31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|03-08-2025|2025|  SUNDAY|      3|         -31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|04-08-2025|2025|  MONDAY|      3|         -32|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|05-08-2025|2025| TUESDAY|      3|         -32|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_bronze.printSchema()

root
 |-- date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
df_silver=df_bronze.withColumn("date" , F.to_date(F.col("date") , "dd-MM-yyyy"))

df_silver.printSchema()

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
df_silver=df_silver.withColumn("day_name" , F.upper(F.col("day_name")))

df_silver = df_silver.dropDuplicates(['date'])
df_silver=df_silver.withColumn("week_of_year" , F.abs(F.col("week_of_year")))

df_silver.show(5)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  FRIDAY|      3|          31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|SATURDAY|      3|          31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  SUNDAY|      3|          31|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-04|2025|  MONDAY|      3|          32|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-05|2025| TUESDAY|      3|          32|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(5)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  FRIDAY|Q3-2025| Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|SATURDAY|Q3-2025| Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  SUNDAY|Q3-2025| Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-04|2025|  MONDAY|Q3-2025| Week32-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-05|2025| TUESDAY|Q3-2025| Week32-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

df_silver.show(3)

+----------+----+--------+-------+-----------+--------------------+--------------------+
|      date|year|day_name|quarter|       week|        _ingested_at|        _source_file|
+----------+----+--------+-------+-----------+--------------------+--------------------+
|2025-08-01|2025|  FRIDAY|Q3-2025|Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|SATURDAY|Q3-2025|Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  SUNDAY|Q3-2025|Week31-2025|2026-02-08 11:22:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+-----------+--------------------+--------------------+
only showing top 3 rows


In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")    